# Yahoo Finance YTD Notebook

Notebook simple: descarga desde Yahoo Finance, procesa YTD y grafica un activo por celda.


In [ ]:
%pip install -q yfinance plotly pandas nbformat ipykernel


In [22]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf

pio.renderers.default = 'vscode'

SERIES = {
    'oil': {'ticker': 'CL=F', 'name': 'WTI Crude Oil', 'unit': 'USD per barrel'},
    'gold': {'ticker': 'GC=F', 'name': 'Gold Futures', 'unit': 'USD per troy ounce'},
    'sp500': {'ticker': '^GSPC', 'name': 'S&P 500', 'unit': 'Index level'},
    'ipsa': {'ticker': '^IPSA', 'name': 'S&P IPSA', 'unit': 'Index level'},
}
START = '1986-01-01'
END = pd.Timestamp.today().strftime('%Y-%m-%d')
ACCENT = '#ffd166'
HISTORICAL = 'rgba(180,180,180,0.26)'


## Utilidades

Funciones para descargar, normalizar y graficar.


In [23]:
def download_series(ticker: str, start: str, end: str) -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)
    if df.empty:
        raise ValueError(f'No data returned for {ticker}')

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
    out = df[[price_col]].rename(columns={price_col: 'price'}).reset_index()
    out.columns = ['date', 'price']
    out['date'] = pd.to_datetime(out['date']).dt.tz_localize(None)
    return out.dropna(subset=['price']).copy()

def build_ytd_frame(df: pd.DataFrame, commodity_key: str, commodity_name: str, unit: str) -> pd.DataFrame:
    data = df.copy()
    data['year'] = data['date'].dt.year
    frames = []
    for year, year_df in data.groupby('year', sort=True):
        year_df = year_df.sort_values('date').reset_index(drop=True).copy()
        if year_df.empty:
            continue
        base_price = year_df.loc[0, 'price']
        year_df['day'] = range(1, len(year_df) + 1)
        year_df['change_pct'] = ((year_df['price'] / base_price) - 1.0) * 100.0
        year_df['commodity_key'] = commodity_key
        year_df['commodity_name'] = commodity_name
        year_df['unit'] = unit
        frames.append(year_df)
    result = pd.concat(frames, ignore_index=True)
    result['is_current'] = result['year'] == result['year'].max()
    return result

def plot_asset(key: str) -> pd.DataFrame:
    meta = SERIES[key]
    raw = download_series(meta['ticker'], START, END)
    ytd = build_ytd_frame(raw, key, meta['name'], meta['unit'])
    current_year = int(ytd['year'].max())
    fig = go.Figure()
    for year, year_df in ytd.groupby('year', sort=True):
        is_current = bool(year == current_year)
        fig.add_trace(go.Scatter(
            x=year_df['day'],
            y=year_df['change_pct'],
            mode='lines',
            name=str(year),
            showlegend=is_current,
            line=dict(color=ACCENT if is_current else HISTORICAL, width=3 if is_current else 1),
            customdata=year_df[['price', 'date']].to_numpy(),
            hovertemplate=(
                '<b>%{fullData.name}</b><br>'
                'Trading day %{x}<br>'
                'YTD %{y:.2f}%<br>'
                'Price %{customdata[0]:,.2f}<br>'
                'Date %{customdata[1]|%Y-%m-%d}'
                '<extra></extra>'
            ),
        ))
    fig.update_layout(
        title=meta['name'],
        paper_bgcolor='#000000',
        plot_bgcolor='#050505',
        font=dict(color='#f2f2f2', family='Arial, sans-serif', size=11),
        xaxis=dict(title='Trading Days', showgrid=False),
        yaxis=dict(title='YTD Change (%)', ticksuffix='%', gridcolor='rgba(255,255,255,0.10)'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        height=420,
        margin=dict(l=50, r=25, t=60, b=40),
    )
    fig.show()
    return ytd


## WTI Crude Oil


In [24]:
oil_ytd = plot_asset('oil')
oil_ytd.tail()


,date,price,year,day,change_pct,commodity_key,commodity_name,unit,is_current
6418,2026-03-18,96.320000,2026,52,68.039079,oil,WTI Crude Oil,USD per barrel,True
6419,2026-03-19,96.139999,2026,53,67.725052,oil,WTI Crude Oil,USD per barrel,True
6420,2026-03-20,98.320000,2026,54,71.528263,oil,WTI Crude Oil,USD per barrel,True
6421,2026-03-23,88.129997,2026,55,53.750868,oil,WTI Crude Oil,USD per barrel,True
6422,2026-03-24,92.349998,2026,56,61.113048,oil,WTI Crude Oil,USD per barrel,True


## Gold Futures


In [26]:
gold_ytd = plot_asset('gold')
gold_ytd.tail()


,date,price,year,day,change_pct,commodity_key,commodity_name,unit,is_current
6409,2026-03-18,4889.899902,2026,52,13.339051,gold,Gold Futures,USD per troy ounce,True
6410,2026-03-19,4600.700195,2026,53,6.635924,gold,Gold Futures,USD per troy ounce,True
6411,2026-03-20,4570.399902,2026,54,5.933618,gold,Gold Futures,USD per troy ounce,True
6412,2026-03-23,4404.100098,2026,55,2.079089,gold,Gold Futures,USD per troy ounce,True
6413,2026-03-24,4399.299805,2026,56,1.967826,gold,Gold Futures,USD per troy ounce,True


## S&P 500


In [27]:
sp500_ytd = plot_asset('sp500')
sp500_ytd.tail()


,date,price,year,day,change_pct,commodity_key,commodity_name,unit,is_current
10129,2026-03-18,6624.700195,2026,52,-3.408486,sp500,S&P 500,Index level,True
10130,2026-03-19,6606.490234,2026,53,-3.673997,sp500,S&P 500,Index level,True
10131,2026-03-20,6506.479980,2026,54,-5.132197,sp500,S&P 500,Index level,True
10132,2026-03-23,6581.000000,2026,55,-4.045658,sp500,S&P 500,Index level,True
10133,2026-03-24,6556.370117,2026,56,-4.404774,sp500,S&P 500,Index level,True


## S&P IPSA


In [28]:
ipsa_ytd = plot_asset('ipsa')
ipsa_ytd.tail()


,date,price,year,day,change_pct,commodity_key,commodity_name,unit,is_current
4341,2019-06-10,5018.990234,2019,111,-2.075555,ipsa,S&P IPSA,Index level,True
4342,2019-06-11,5069.240234,2019,112,-1.095138,ipsa,S&P IPSA,Index level,True
4343,2019-06-12,5067.850098,2019,113,-1.122261,ipsa,S&P IPSA,Index level,True
4344,2019-06-13,5071.759766,2019,114,-1.045980,ipsa,S&P IPSA,Index level,True
4345,2019-06-14,5058.879883,2019,115,-1.297277,ipsa,S&P IPSA,Index level,True
